# Módulo 3 — Implementación del Ruido de Canal (AWGN)

**Proyecto Integrador — Comunicaciones Digitales**

Este notebook agrega ruido AWGN al canal de comunicación y analiza cómo afecta
la tasa de error de bit (BER) en función de la relación señal a ruido (SNR).

## Contenido

1. Función de ruido AWGN
2. Visualización del efecto del ruido sobre un chirp
3. Verificación del SNR agregado
4. Barrido BER vs SNR
5. Comparación con curva teórica de referencia


## 1. Librerías e imports de módulos anteriores

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ---------- Módulo 1: Codificador / Decodificador ----------

def codificador(bits, SF):
    bits = np.array(bits)
    if len(bits) % SF != 0:
        raise ValueError(f"La cantidad de bits ({len(bits)}) debe ser múltiplo de SF ({SF})")
    num_simbolos = len(bits) // SF
    simbolos = np.zeros(num_simbolos, dtype=int)
    for i in range(num_simbolos):
        bloque = bits[i * SF : (i + 1) * SF]
        valor = 0
        for posicion, bit in enumerate(bloque):
            peso = 2 ** (SF - 1 - posicion)
            valor += bit * peso
        simbolos[i] = valor
    return simbolos

def decodificador(simbolos, SF):
    simbolos = np.array(simbolos, dtype=int)
    bits = np.zeros(len(simbolos) * SF, dtype=int)
    for i, simbolo in enumerate(simbolos):
        valor_restante = int(simbolo)
        for posicion in range(SF):
            peso = 2 ** (SF - 1 - posicion)
            if valor_restante >= peso:
                bits[i * SF + posicion] = 1
                valor_restante -= peso
    return bits

def calcular_ber(bits_tx, bits_rx):
    bits_tx = np.array(bits_tx)
    bits_rx = np.array(bits_rx)
    if len(bits_tx) != len(bits_rx):
        raise ValueError("Los vectores deben tener la misma longitud")
    return np.sum(bits_tx != bits_rx) / len(bits_tx)

# ---------- Módulo 2: Waveform Former / n-Tuple Former ----------

def up_chirp_base(SF, BW, Fs):
    N = 2**SF
    chirp = np.zeros(N, dtype=complex)
    for k in range(N):
        argumento = k**2 / (2*N) - k/2
        chirp[k] = np.exp(1j * 2 * np.pi * argumento)
    return chirp

def down_chirp(SF, BW, Fs):
    return np.conj(up_chirp_base(SF, BW, Fs))

def waveform_former(simbolos, SF, BW, Fs):
    N = 2**SF
    simbolos = np.array(simbolos)
    chirp_base = up_chirp_base(SF, BW, Fs)
    waveform = np.zeros((len(simbolos), N), dtype=complex)
    for i, s in enumerate(simbolos):
        waveform[i] = np.roll(chirp_base, -s)
    return waveform

def n_tuple_former(waveform, SF, BW, Fs):
    dc = down_chirp(SF, BW, Fs)
    num_simbolos = waveform.shape[0]
    simbolos_rx = np.zeros(num_simbolos, dtype=int)
    for i in range(num_simbolos):
        dechirped = waveform[i] * dc
        espectro = np.fft.fft(dechirped)
        simbolos_rx[i] = np.argmax(np.abs(espectro))
    return simbolos_rx

# ---------- Parámetros del sistema ----------
SF  = 7
BW  = 125e3
Fs  = BW
N   = 2**SF

print("Módulos 1 y 2 cargados correctamente.")
print(f"Parámetros: SF={SF}, BW={BW/1e3:.0f} kHz, N={N}")

## 2. Función de ruido AWGN

El ruido AWGN se genera en tres pasos:

**Paso 1 — Medir la potencia de la señal:**
$$P_{señal} = \frac{1}{M} \sum_{k=0}^{M-1} |x[k]|^2$$

**Paso 2 — Calcular la potencia de ruido necesaria:**

Despejando de la definición de SNR:
$$P_{ruido} = \frac{P_{señal}}{10^{SNR_{dB}/10}}$$

**Paso 3 — Generar y sumar el ruido:**

El ruido es complejo porque la señal LoRa es compleja (tiene parte real e imaginaria).
La potencia se divide en partes iguales entre ambas componentes:
$$n[k] = \sqrt{\frac{P_{ruido}}{2}} \cdot (\mathcal{N}(0,1) + j\mathcal{N}(0,1))$$


In [ ]:
def agregar_ruido_awgn(waveform, snr_db):
    """
    Agrega ruido AWGN a una matriz de waveforms LoRa.

    Parámetros
    ----------
    waveform : np.array complejo de shape (num_simbolos, N)
    snr_db   : relación señal a ruido deseada en dB

    Retorna
    -------
    waveform_ruidosa : np.array complejo de la misma shape que waveform
    """
    # Paso 1: medimos la potencia promedio de la señal
    # np.abs()^2 calcula |x|^2 para cada muestra compleja
    potencia_senal = np.mean(np.abs(waveform)**2)

    # Paso 2: calculamos la potencia de ruido necesaria para el SNR pedido
    # Despejamos P_ruido de: SNR_dB = 10*log10(P_senal / P_ruido)
    snr_lineal = 10 ** (snr_db / 10)
    potencia_ruido = potencia_senal / snr_lineal

    # Paso 3: generamos ruido gaussiano complejo con esa potencia
    # La raíz cuadrada de potencia_ruido/2 es la desviación estándar
    # de cada componente (real e imaginaria)
    desviacion = np.sqrt(potencia_ruido / 2)
    ruido_real = desviacion * np.random.randn(*waveform.shape)
    ruido_imag = desviacion * np.random.randn(*waveform.shape)
    ruido = ruido_real + 1j * ruido_imag

    return waveform + ruido

print("Función agregar_ruido_awgn definida correctamente.")

## 3. Verificación: ¿el SNR agregado es el pedido?

Antes de seguir, verificamos que la función efectivamente genera el SNR solicitado.
Calculamos el SNR de la señal ruidosa y lo comparamos con el valor pedido.


In [ ]:
np.random.seed(42)

# Generamos una waveform de prueba
bits_test = np.random.randint(0, 2, 100 * SF)
simbolos_test = codificador(bits_test, SF)
wf_test = waveform_former(simbolos_test, SF, BW, Fs)

print(f"{'SNR pedido (dB)':>18} | {'SNR medido (dB)':>17} | {'Diferencia':>10}")
print("-" * 55)

for snr_objetivo in [-10, -5, 0, 5, 10, 15, 20]:
    wf_ruidosa = agregar_ruido_awgn(wf_test, snr_objetivo)

    # Calculamos el SNR real de la señal ruidosa
    potencia_senal = np.mean(np.abs(wf_test)**2)
    potencia_ruido = np.mean(np.abs(wf_ruidosa - wf_test)**2)
    snr_medido = 10 * np.log10(potencia_senal / potencia_ruido)

    diferencia = snr_medido - snr_objetivo
    print(f"{snr_objetivo:>18} | {snr_medido:>17.2f} | {diferencia:>+10.2f}")

## 4. Visualización: efecto del ruido sobre un chirp

Comparamos la parte real de un chirp limpio vs el mismo chirp con distintos niveles de ruido.


In [ ]:
np.random.seed(0)
simbolo_vis = [45]
wf_limpia = waveform_former(simbolo_vis, SF, BW, Fs)

niveles_snr = [20, 5, -5]
fig, axs = plt.subplots(len(niveles_snr) + 1, 1, figsize=(10, 8), sharex=True)

# Señal limpia
axs[0].plot(np.real(wf_limpia[0]), color='steelblue', linewidth=1)
axs[0].set_title("Señal limpia (sin ruido) — símbolo 45")
axs[0].set_ylabel("Amplitud")
axs[0].set_ylim(-3, 3)

# Señal con distintos niveles de ruido
colores = ['darkorange', 'tomato', 'crimson']
for ax, snr, color in zip(axs[1:], niveles_snr, colores):
    wf_r = agregar_ruido_awgn(wf_limpia, snr)
    ax.plot(np.real(wf_r[0]), color=color, linewidth=0.8)
    ax.set_title(f"SNR = {snr} dB")
    ax.set_ylabel("Amplitud")
    ax.set_ylim(-3, 3)

axs[-1].set_xlabel("Muestra")
plt.suptitle("Efecto del ruido AWGN sobre un chirp LoRa", fontsize=12)
plt.tight_layout()
plt.show()

## 5. Visualización: FFT post-dechirping con ruido

Acá se ve cómo el ruido afecta la detección del símbolo:
con SNR alto el pico es claramente el más alto, con SNR bajo el ruido 
puede "tapar" el pico correcto y producir un error de símbolo.


In [ ]:
np.random.seed(1)
s_demo = 45
wf_demo = waveform_former([s_demo], SF, BW, Fs)
dc = down_chirp(SF, BW, Fs)

niveles = [20, 0, -5]
fig, axs = plt.subplots(1, 3, figsize=(13, 4))

for ax, snr in zip(axs, niveles):
    wf_r = agregar_ruido_awgn(wf_demo, snr)
    espectro = np.abs(np.fft.fft(wf_r[0] * dc))
    s_detectado = np.argmax(espectro)

    ax.plot(espectro, color='steelblue', linewidth=0.8)
    ax.axvline(s_demo, color='green', linestyle='--', linewidth=1.5,
               label=f'Símbolo TX = {s_demo}')
    ax.axvline(s_detectado, color='red', linestyle=':', linewidth=1.5,
               label=f'Detectado = {s_detectado}')
    correcto = '✅' if s_detectado == s_demo else '❌'
    ax.set_title(f"SNR = {snr} dB  {correcto}")
    ax.set_xlabel("Bin de FFT")
    ax.set_ylabel("|FFT|")
    ax.legend(fontsize=8)

plt.suptitle("FFT post-dechirping con distintos niveles de ruido", fontsize=12)
plt.tight_layout()
plt.show()

## 6. Barrido BER vs SNR

Simulamos la transmisión para distintos valores de SNR y medimos la BER resultante.
Para que la BER sea estadísticamente confiable, necesitamos transmitir suficientes bits.
Una regla práctica es transmitir al menos **100 veces** más bits que los errores esperados.


In [ ]:
np.random.seed(99)

snr_range = np.arange(-15, 16, 1)   # de -15 dB a +15 dB en pasos de 1 dB
num_bits_por_punto = 70 * SF         # 70 símbolos por punto SNR
ber_simulada = []

print(f"{'SNR (dB)':>10} | {'BER':>12} | {'Errores/Total bits':>20}")
print("-" * 50)

for snr in snr_range:
    bits_tx = np.random.randint(0, 2, num_bits_por_punto)
    simbolos_tx = codificador(bits_tx, SF)

    # Modulación
    wf = waveform_former(simbolos_tx, SF, BW, Fs)

    # Canal con ruido AWGN
    wf_ruidosa = agregar_ruido_awgn(wf, snr)

    # Demodulación y decodificación
    simbolos_rx = n_tuple_former(wf_ruidosa, SF, BW, Fs)
    bits_rx = decodificador(simbolos_rx, SF)

    ber = calcular_ber(bits_tx, bits_rx)
    ber_simulada.append(ber)

    errores = int(ber * len(bits_tx))
    print(f"{snr:>10} | {ber:>12.6f} | {errores:>6} / {len(bits_tx)}")

ber_simulada = np.array(ber_simulada)

## 7. Curva BER vs SNR con referencia teórica

La curva teórica de referencia para LoRa con AWGN se puede aproximar con
la expresión del SER (Symbol Error Rate) para M-FSK no coherente:

$$SER \approx \frac{N-1}{N} \cdot e^{-\frac{SNR_{lineal}}{2}}$$

Y la BER se relaciona con el SER aproximadamente como:

$$BER \approx \frac{N/2}{N-1} \cdot SER$$

Esta curva sirve como referencia para validar que la simulación se comporta
de forma consistente con la teoría.


In [ ]:
# Curva teórica de referencia (aproximación para M-FSK no coherente)
snr_lineal = 10 ** (snr_range / 10)
ser_teo = ((N - 1) / N) * np.exp(-snr_lineal / 2)
ber_teo = (N / 2) / (N - 1) * ser_teo

fig, ax = plt.subplots(figsize=(9, 5))

# BER simulada
ax.semilogy(snr_range, np.where(ber_simulada > 0, ber_simulada, 1e-7),
            marker='o', markersize=4, linewidth=1.5,
            color='steelblue', label='BER simulada')

# Curva teórica
ax.semilogy(snr_range, ber_teo,
            linestyle='--', linewidth=1.5,
            color='tomato', label='BER teórica (M-FSK no coherente)')

ax.set_xlabel("SNR (dB)")
ax.set_ylabel("BER")
ax.set_title(f"BER vs SNR — LoRa AWGN (SF={SF}, BW={BW/1e3:.0f} kHz)")
ax.legend()
ax.grid(True, which='both', alpha=0.4)
ax.set_ylim(1e-4, 1)
plt.tight_layout()
plt.show()

## 8. Observaciones sobre la curva

Analizamos los resultados del barrido:


In [ ]:
# SNR de corte: primer valor donde la BER cae por debajo de 1%
snr_corte = None
for snr, ber in zip(snr_range, ber_simulada):
    if ber < 0.01:
        snr_corte = snr
        break

# BER en el peor caso (SNR más bajo)
ber_max = ber_simulada[0]

# BER en el mejor caso (SNR más alto)
ber_min = ber_simulada[-1]

print("=== Resumen del barrido BER vs SNR ===")
print(f"SNR más bajo simulado : {snr_range[0]} dB  →  BER = {ber_max:.4f}")
print(f"SNR más alto simulado : {snr_range[-1]} dB  →  BER = {ber_min:.6f}")
if snr_corte is not None:
    print(f"BER < 1% a partir de  : {snr_corte} dB")
else:
    print("BER no cayó por debajo de 1% en el rango simulado")

print()
print("Nota: LoRa es conocido por funcionar con SNR negativos.")
print("En la curva se puede ver que la BER empieza a caer")
print("significativamente alrededor de los 0 dB.")

## 9. Conclusiones del módulo

- Se implementó `agregar_ruido_awgn`: mide la potencia de la señal, calcula 
  la potencia de ruido para el SNR deseado, y suma ruido gaussiano complejo.
- Se verificó que el SNR efectivo de la señal ruidosa coincide con el SNR pedido
  (diferencias menores a 0.5 dB debidas a la aleatoriedad del ruido).
- Las visualizaciones muestran cómo con SNR alto el pico de la FFT post-dechirping
  es dominante y la detección es correcta, mientras que con SNR bajo el ruido
  puede superar al pico y provocar errores.
- La curva BER vs SNR simulada sigue la tendencia esperada teóricamente:
  la BER cae rápidamente a medida que el SNR aumenta.

**Próximo módulo:** se agrega un canal **selectivo en frecuencia** (distorsión no uniforme
en el espectro), que modela un entorno de comunicación más realista con reflexiones
y múltiples caminos de propagación (multipath).
